# 15 RAG、评测和服务化面试链路

目标：用一个轻量级检索例子串起 RAG 的关键问题：chunk、retrieve、context budget、引用、评测指标，以及线上服务常问的 TTFT、TPOT、吞吐和缓存。

这个 notebook 不依赖向量库，先用可解释的 TF-IDF 检索练流程。真实生产可以把检索器替换成 embedding + vector database + reranker。


## 1. 准备一个小知识库


In [ ]:
import math
import re
from collections import Counter, defaultdict


documents = [
    {
        "id": "kv-cache",
        "source": "deployment-notes.md#kv-cache",
        "text": "KV cache 保存每层 attention 的 key 和 value。prefill 会为整段 prompt 建缓存，decode 每生成一个 token 追加一格缓存。长上下文和高并发会显著增加显存占用。",
    },
    {
        "id": "ttft-tpot",
        "source": "deployment-notes.md#latency",
        "text": "TTFT 表示 time to first token，主要受排队、tokenize、prefill 和调度影响。TPOT 表示 time per output token，常用于衡量 decode 阶段速度。",
    },
    {
        "id": "quantization",
        "source": "deployment-notes.md#quantization",
        "text": "量化把权重从 FP16 或 FP32 降到 INT8、INT4 等格式，通常降低显存占用。速度是否提升取决于硬件 kernel、batch 大小和反量化开销。",
    },
    {
        "id": "lora",
        "source": "deployment-notes.md#lora",
        "text": "LoRA 冻结基础模型，只训练低秩 adapter。它能降低微调显存和保存成本，但线上需要管理 adapter 加载、合并和多租户隔离。",
    },
]

for doc in documents:
    print(doc["id"], "->", doc["text"])


## 2. 一个可解释的 TF-IDF 检索器

面试时可以先讲清楚检索流程，再说明生产里常用 dense embedding、hybrid search 和 reranker。


In [ ]:
def tokenize(text):
    # 中文按单字切，英文和数字按词切。这个函数只用于教学，不代表最佳中文检索方案。
    return re.findall(r"[一-鿿]|[A-Za-z0-9_]+", text.lower())


def build_index(docs):
    doc_tokens = {doc["id"]: tokenize(doc["text"]) for doc in docs}
    df = Counter()
    for tokens in doc_tokens.values():
        df.update(set(tokens))

    total_docs = len(docs)
    idf = {term: math.log((total_docs + 1) / (freq + 1)) + 1 for term, freq in df.items()}
    vectors = {}
    for doc in docs:
        counts = Counter(doc_tokens[doc["id"]])
        vectors[doc["id"]] = {term: count * idf.get(term, 0.0) for term, count in counts.items()}
    return idf, vectors


def cosine(a, b):
    shared = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in shared)
    norm_a = math.sqrt(sum(v * v for v in a.values()))
    norm_b = math.sqrt(sum(v * v for v in b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


idf, doc_vectors = build_index(documents)
doc_by_id = {doc["id"]: doc for doc in documents}


def vectorize_query(query):
    counts = Counter(tokenize(query))
    return {term: count * idf.get(term, 0.0) for term, count in counts.items()}


def retrieve(query, k=2):
    query_vector = vectorize_query(query)
    hits = []
    for doc_id, vector in doc_vectors.items():
        hits.append((cosine(query_vector, vector), doc_by_id[doc_id]))
    hits.sort(key=lambda item: item[0], reverse=True)
    return hits[:k]


query = "为什么长上下文并发会让显存变大？"
for score, doc in retrieve(query, k=3):
    print(f"score={score:.3f}", doc["id"], doc["source"])
    print(doc["text"])


## 3. chunk 和 context budget

RAG 不是把所有资料都塞进 prompt。需要控制 chunk 粒度、overlap、top-k、重排和最终上下文预算。


In [ ]:
def chunk_text(text, max_chars=42, overlap=10):
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks


long_text = documents[0]["text"] + documents[1]["text"]
chunks = chunk_text(long_text)
for index, chunk in enumerate(chunks):
    print(index, repr(chunk))


In [ ]:
def build_rag_prompt(question, hits, max_context_chars=260):
    context_blocks = []
    used = 0
    for rank, (score, doc) in enumerate(hits, start=1):
        block = f"[{rank}] source={doc['source']}\n{doc['text']}"
        if used + len(block) > max_context_chars:
            continue
        context_blocks.append(block)
        used += len(block)

    context = "\n\n".join(context_blocks)
    return (
        "你是一个严谨的问答助手。只能根据给定资料回答，答案里必须带引用编号。\n\n"
        f"资料：\n{context}\n\n"
        f"问题：{question}\n"
        "答案："
    )


question = "TTFT 和 TPOT 分别受什么影响？"
hits = retrieve(question, k=3)
print(build_rag_prompt(question, hits))


## 4. 检索评测：Recall@k 和 MRR

生成质量差不一定是模型差，可能是检索没召回、chunk 切坏、rerank 错误或上下文超预算。


In [ ]:
eval_set = [
    {"question": "KV cache 为什么会吃显存？", "gold_doc": "kv-cache"},
    {"question": "首 token 延迟叫什么，主要和什么有关？", "gold_doc": "ttft-tpot"},
    {"question": "INT4 量化一定更快吗？", "gold_doc": "quantization"},
    {"question": "LoRA 为什么适合低成本微调？", "gold_doc": "lora"},
]


def evaluate_retrieval(dataset, k=2):
    recall_hits = 0
    reciprocal_ranks = []
    for row in dataset:
        ranked = retrieve(row["question"], k=len(documents))
        ranked_ids = [doc["id"] for _, doc in ranked]
        if row["gold_doc"] in ranked_ids[:k]:
            recall_hits += 1
        rank = ranked_ids.index(row["gold_doc"]) + 1 if row["gold_doc"] in ranked_ids else None
        reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)
        print(row["question"], "gold=", row["gold_doc"], "ranked=", ranked_ids)

    recall_at_k = recall_hits / len(dataset)
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    print(f"Recall@{k} = {recall_at_k:.2f}")
    print(f"MRR = {mrr:.2f}")


evaluate_retrieval(eval_set, k=2)


## 5. 引用和 groundedness 的最小检查

真实系统需要更严格的事实一致性评测。这里先做一个最低限度检查：答案引用的资料编号必须存在于 context。


In [ ]:
def extract_citations(answer):
    return [int(x) for x in re.findall(r"\[(\d+)\]", answer)]


def check_citations(answer, hits):
    valid = set(range(1, len(hits) + 1))
    cited = extract_citations(answer)
    missing = [number for number in cited if number not in valid]
    return {"cited": cited, "missing": missing, "ok": bool(cited) and not missing}


answer = "TTFT 是首 token 延迟，主要受排队、tokenize、prefill 和调度影响；TPOT 衡量 decode 阶段每个输出 token 的耗时。[1]"
print(check_citations(answer, hits))


## 6. 服务化指标：TTFT、TPOT、吞吐

线上排查要把请求拆成排队、prefill、decode、网络和后处理。只看总延迟无法判断瓶颈。


In [ ]:
requests = [
    {"id": "r1", "prompt_tokens": 256, "output_tokens": 80, "queue_ms": 12},
    {"id": "r2", "prompt_tokens": 2048, "output_tokens": 40, "queue_ms": 35},
    {"id": "r3", "prompt_tokens": 512, "output_tokens": 160, "queue_ms": 20},
]

prefill_ms_per_token = 0.08
decode_ms_per_token = 18.0
network_ms = 25

for req in requests:
    prefill_ms = req["prompt_tokens"] * prefill_ms_per_token
    decode_ms = req["output_tokens"] * decode_ms_per_token
    ttft = req["queue_ms"] + prefill_ms + network_ms
    latency = ttft + decode_ms
    tpot = decode_ms / max(req["output_tokens"], 1)
    throughput = req["output_tokens"] / (latency / 1000)
    print(
        req["id"],
        f"TTFT={ttft:.1f}ms",
        f"TPOT={tpot:.1f}ms/token",
        f"latency={latency:.1f}ms",
        f"throughput={throughput:.1f} out_tok/s",
    )


## 面试总结

- RAG 链路：query rewrite -> retrieve -> rerank -> context assemble -> generate -> cite -> evaluate。
- 检索评测先看 Recall@k、MRR、NDCG；生成评测再看 groundedness、faithfulness、引用正确率和人工偏好。
- chunk 过短会丢语义，过长会降低召回精度并浪费 context budget。
- 线上服务指标要分开看 TTFT、TPOT、总延迟、吞吐、排队时间、错误率和 GPU 利用率。
- 常见优化：embedding/cache、prefix cache、continuous batching、paged attention、量化、speculative decoding、异步队列和限流。
- 面试中不要只说“加 RAG 可以减少幻觉”，要能说明检索失败、资料过期、引用错配和上下文污染这些风险。
